# Kudge Case Study Extraction

Extract compact case-study candidates from the Kudge solver-vs-judge K-factor comparison.

Categories:

- hard to solve + hard to judge
- easy to solve + hard to judge
- hard to solve + easy to judge

Difficulty is based on percentile ranks of centered K-factor item difficulty.

In [ ]:
from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 180)


In [ ]:
K = 2
TOP_N = 10
HIGH_PCT = 75
LOW_PCT = 25

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "K-Factor").exists():
    REPO_ROOT = next(p for p in Path.cwd().parents if (p / "K-Factor").exists())

paired_path = REPO_ROOT / "K-Factor" / "results" / "kudge_solver_judge_comparison" / f"kudge_k{K}_paired_items.csv"
percentile_path = REPO_ROOT / "K-Factor" / "results" / "kudge_solver_judge_comparison" / f"kudge_k{K}_solver_judge_difficulty_percentiles.csv"
out_dir = REPO_ROOT / "K-Factor" / "paper_artifacts" / "kudge" / "case_studies"
out_dir.mkdir(parents=True, exist_ok=True)

print(paired_path, paired_path.exists())
print(percentile_path, percentile_path.exists())
print(out_dir)


In [ ]:
def parse_score_dict(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, SyntaxError):
        return {}


def mean_dict_score(value):
    scores = parse_score_dict(value)
    vals = [float(v) for v in scores.values() if v is not None and not pd.isna(v)]
    return float(np.mean(vals)) if vals else np.nan


paired = pd.read_csv(paired_path)
percentiles = pd.read_csv(percentile_path)[[
    "solver_item_id",
    "solver_difficulty_percentile",
    "judge_difficulty_percentile",
]]

items = paired.merge(percentiles, on="solver_item_id", how="left", validate="one_to_one")
items["solver_score_mean"] = items["solver_scores"].apply(mean_dict_score)
items["judge_score_mean"] = items["judge_scores"].apply(mean_dict_score)

print(f"items: {items.shape}")
print(f"missing solver percentiles: {items['solver_difficulty_percentile'].isna().sum()}")
print(f"missing judge percentiles:  {items['judge_difficulty_percentile'].isna().sum()}")

display(items[[
    "solver_item_id",
    "solver_subset",
    "solver_difficulty_centered",
    "solver_difficulty_percentile",
    "judge_difficulty_centered",
    "judge_difficulty_percentile",
    "solver_score_mean",
    "judge_score_mean",
]].head())


In [ ]:
def select_case_studies(df, category, mask, score_fn, top_n=TOP_N):
    selected = df.loc[mask].copy()
    selected["case_category"] = category
    selected["case_rank_score"] = score_fn(selected)
    return selected.sort_values("case_rank_score", ascending=False).head(top_n)


hard_solve_hard_judge = select_case_studies(
    items,
    "hard_to_solve__hard_to_judge",
    (items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT),
    lambda d: np.minimum(d["solver_difficulty_percentile"], d["judge_difficulty_percentile"]),
)

easy_solve_hard_judge = select_case_studies(
    items,
    "easy_to_solve__hard_to_judge",
    (items["solver_difficulty_percentile"] <= LOW_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT),
    lambda d: np.minimum(100 - d["solver_difficulty_percentile"], d["judge_difficulty_percentile"]),
)

hard_solve_easy_judge = select_case_studies(
    items,
    "hard_to_solve__easy_to_judge",
    (items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] <= LOW_PCT),
    lambda d: np.minimum(d["solver_difficulty_percentile"], 100 - d["judge_difficulty_percentile"]),
)

case_studies = pd.concat(
    [hard_solve_hard_judge, easy_solve_hard_judge, hard_solve_easy_judge],
    ignore_index=True,
)

summary = pd.DataFrame([
    {
        "case_category": "hard_to_solve__hard_to_judge",
        "n_available": int(((items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT)).sum()),
        "n_selected": len(hard_solve_hard_judge),
    },
    {
        "case_category": "easy_to_solve__hard_to_judge",
        "n_available": int(((items["solver_difficulty_percentile"] <= LOW_PCT) & (items["judge_difficulty_percentile"] >= HIGH_PCT)).sum()),
        "n_selected": len(easy_solve_hard_judge),
    },
    {
        "case_category": "hard_to_solve__easy_to_judge",
        "n_available": int(((items["solver_difficulty_percentile"] >= HIGH_PCT) & (items["judge_difficulty_percentile"] <= LOW_PCT)).sum()),
        "n_selected": len(hard_solve_easy_judge),
    },
])

display(summary)

display(case_studies[[
    "case_category",
    "case_rank_score",
    "solver_item_id",
    "solver_subset",
    "solver_difficulty_percentile",
    "judge_difficulty_percentile",
    "solver_difficulty_centered",
    "judge_difficulty_centered",
    "solver_score_mean",
    "judge_score_mean",
]].sort_values(["case_category", "case_rank_score"], ascending=[True, False]))


In [ ]:
case_columns = [
    "case_category",
    "case_rank_score",
    "solver_item_id",
    "solver_subset",
    "solver_gold",
    "solver_difficulty_centered",
    "solver_difficulty_centered_laplace_se",
    "solver_difficulty_percentile",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
    "judge_difficulty_percentile",
    "solver_score_mean",
    "judge_score_mean",
    "solver_prompt",
    "solver_chosen",
    "solver_rejected",
    "solver_scores",
    "judge_scores",
]

compact_columns = [
    "case_category",
    "case_rank_score",
    "solver_item_id",
    "solver_subset",
    "solver_gold",
    "solver_difficulty_centered",
    "solver_difficulty_centered_laplace_se",
    "solver_difficulty_percentile",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
    "judge_difficulty_percentile",
    "solver_score_mean",
    "judge_score_mean",
]

case_studies_full = case_studies[case_columns].copy()
case_studies_compact = case_studies[compact_columns].copy()

case_studies_full.to_csv(out_dir / f"kudge_k{K}_case_studies_full.csv", index=False)
case_studies_full.to_json(out_dir / f"kudge_k{K}_case_studies_full.json", orient="records", indent=2, force_ascii=False)
case_studies_compact.to_csv(out_dir / f"kudge_k{K}_case_studies_compact.csv", index=False)
case_studies_compact.to_json(out_dir / f"kudge_k{K}_case_studies_compact.json", orient="records", indent=2, force_ascii=False)
summary.to_csv(out_dir / f"kudge_k{K}_case_study_summary.csv", index=False)
summary.to_json(out_dir / f"kudge_k{K}_case_study_summary.json", orient="records", indent=2, force_ascii=False)

print("Saved:")
for path in sorted(out_dir.glob(f"kudge_k{K}_case_stud*.csv")):
    print(path.relative_to(REPO_ROOT))
for path in sorted(out_dir.glob(f"kudge_k{K}_case_stud*.json")):
    print(path.relative_to(REPO_ROOT))


In [ ]:
# Optional: inspect one category at a time without opening the full text-heavy file.
category = "hard_to_solve__hard_to_judge"
display(
    case_studies_compact
    .loc[case_studies_compact["case_category"] == category]
    .sort_values("case_rank_score", ascending=False)
)
